In [ ]:
from pathlib import Path

import cv2
import numpy as np


# ============================ CONFIGURACION ==============================

VIDEO = Path("Videos") / "video30min-11to22.mp4"
SALIDA = Path("Videos")

FPS_SALIDA = 5.0

# Inicio incluido y final excluido.
SEGUNDO_INICIO = None
SEGUNDO_FIN = 30


# ------------------------------------------------------------------------
# Atlas temporal: solamente para textos fijos
# ------------------------------------------------------------------------

MUESTRAS_ATLAS_TEXTO = 400
UMBRAL_ATLAS_TEXTO = 0.12
RADIO_ATLAS_TEXTO = 6


# ------------------------------------------------------------------------
# Deteccion de verde
# Formato:
# H minimo, H maximo, S minimo, V minimo, G-R minimo, G-B minimo
# ------------------------------------------------------------------------

VERDE_FUERTE = (
    37,
    90,
    105,
    55,
    15,
    12
)

VERDE_MEDIO = (
    32,
    100,
    38,
    30,
    6,
    5
)

# Solo se usa para medir el crosshair.
# Nunca se copia directamente a la mascara final.
VERDE_TENUE = (
    28,
    108,
    16,
    16,
    2,
    2
)


# ------------------------------------------------------------------------
# Continuidad temporal
# ------------------------------------------------------------------------

RADIO_TEMPORAL = 4
RADIO_ALINEACION = 4
MINIMO_VECINOS = 1


# ------------------------------------------------------------------------
# Crosshair paramétrico corregido
# ------------------------------------------------------------------------

DX_INICIAL = 0.075
DY_INICIAL = 0.080

DX_MINIMO = 0.045
DX_MAXIMO = 0.145

DY_MINIMO = 0.045
DY_MAXIMO = 0.180

# Se adapta más rápido a cambios reales de apertura.
PESO_APERTURA_ANTERIOR = 0.60

MAXIMO_CAMBIO_X = 0.020
MAXIMO_CAMBIO_Y = 0.025

MINIMO_PIXELES_ESQUINA = 2

# Ventana utilizada para encontrar la intersección de cada L.
LONGITUD_BUSQUEDA_ESQUINA = 24
GROSOR_BUSQUEDA_ESQUINA = 3

# Las patas se dibujan ligeramente mayores que las observadas.
LONGITUD_PATA_HORIZONTAL_RELATIVA = 0.024
LONGITUD_PATA_VERTICAL_RELATIVA = 0.032

GROSOR_PLANTILLA = 3
MARGEN_PLANTILLA = 2
DILATACION_CROSSHAIR = 5

# Brazos centrales.
PROPORCION_BRAZO_HORIZONTAL = 0.67
PROPORCION_BRAZO_VERTICAL = 0.61

HUECO_CENTRAL = 3

# X central.
RADIO_X_CENTRAL = 7
GROSOR_X_CENTRAL = 3

# Dos marcas verticales laterales.
POSICION_MARCA_LATERAL = 0.82
LONGITUD_MARCA_LATERAL_RELATIVA = 0.080
GROSOR_MARCA_LATERAL = 3


# ------------------------------------------------------------------------
# Medicion geometrica
# ------------------------------------------------------------------------

LONGITUDES_LINEA = (
    3,
    5,
    7,
    9,
    13,
    17,
    23
)

USAR_LSD = True
TOLERANCIA_ANGULO = 14.0

LONGITUD_MINIMA_LSD = 3.0
LONGITUD_MAXIMA_LSD = 190.0


# ------------------------------------------------------------------------
# Texto, brujula e instrumentos inferiores
# ------------------------------------------------------------------------

CIERRE_TEXTO = 3
DILATACION_TEXTO = 3

RADIO_ANCLA_ESTRICTA = 5
RADIO_CRECIMIENTO_ESTRICTO = 3
CIERRE_ESTRICTO = 3
DILATACION_ESTRICTA = 3


# ------------------------------------------------------------------------
# Archivos
# ------------------------------------------------------------------------

COMPRESION_PNG = 3

GUARDAR_ATLAS_TEXTO = True
GUARDAR_VISTAS = True
GUARDAR_DIAGNOSTICOS = True

LIMPIAR_ANTERIORES = True

# ========================================================================


def convertir_a_impar(valor):
    valor = max(
        1,
        int(valor)
    )

    if valor % 2 == 0:
        valor += 1

    return valor


def calcular_digitos(cantidad):
    return len(
        str(
            max(
                0,
                cantidad - 1
            )
        )
    )


def obtener_datos_video(captura):
    fps = float(
        captura.get(
            cv2.CAP_PROP_FPS
        )
    )

    cantidad_frames = int(
        captura.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if fps <= 0:
        raise RuntimeError(
            "No fue posible obtener los FPS del video."
        )

    if cantidad_frames <= 0:
        raise RuntimeError(
            "No fue posible obtener la cantidad de frames."
        )

    duracion = cantidad_frames / fps

    return (
        fps,
        cantidad_frames,
        duracion
    )


def calcular_intervalo(duracion_video):
    if SEGUNDO_INICIO is None:
        inicio = 0.0
    else:
        inicio = float(
            SEGUNDO_INICIO
        )

    if SEGUNDO_FIN is None:
        fin = float(
            duracion_video
        )
    else:
        fin = float(
            SEGUNDO_FIN
        )

    inicio = max(
        0.0,
        inicio
    )

    fin = min(
        float(duracion_video),
        fin
    )

    if inicio >= duracion_video:
        raise ValueError(
            f"El inicio es {inicio:.3f}, pero el video dura "
            f"{duracion_video:.3f} segundos."
        )

    if fin <= inicio:
        raise ValueError(
            f"El final ({fin:.3f}) debe ser mayor "
            f"que el inicio ({inicio:.3f})."
        )

    return inicio, fin


def crear_tiempos_salida(inicio, fin):
    cantidad = max(
        1,
        int(
            np.ceil(
                (fin - inicio)
                * FPS_SALIDA
                - 1e-9
            )
        )
    )

    tiempos = (
        inicio
        + np.arange(
            cantidad,
            dtype=np.float64
        ) / FPS_SALIDA
    )

    tiempos = tiempos[
        tiempos < fin - 1e-9
    ]

    if len(tiempos) == 0:
        tiempos = np.array(
            [inicio],
            dtype=np.float64
        )

    return tiempos


def obtener_canales(frame):
    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    azul, verde, rojo = cv2.split(
        frame
    )

    return (
        hsv,
        azul.astype(np.int16),
        verde.astype(np.int16),
        rojo.astype(np.int16)
    )


def detectar_verde(
    frame,
    configuracion
):
    (
        tono_minimo,
        tono_maximo,
        saturacion_minima,
        brillo_minimo,
        diferencia_rojo,
        diferencia_azul
    ) = configuracion

    hsv, azul, verde, rojo = obtener_canales(
        frame
    )

    mascara_hsv = cv2.inRange(
        hsv,
        np.array(
            [
                tono_minimo,
                saturacion_minima,
                brillo_minimo
            ],
            dtype=np.uint8
        ),
        np.array(
            [
                tono_maximo,
                255,
                255
            ],
            dtype=np.uint8
        )
    )

    dominancia = (
        (verde >= rojo + diferencia_rojo)
        & (verde >= azul + diferencia_azul)
    ).astype(np.uint8) * 255

    return cv2.bitwise_and(
        mascara_hsv,
        dominancia
    )


def agregar_rectangulo_relativo(
    mascara,
    x1,
    y1,
    x2,
    y2
):
    alto, ancho = mascara.shape

    izquierda = int(
        np.clip(
            round(ancho * x1),
            0,
            ancho - 1
        )
    )

    superior = int(
        np.clip(
            round(alto * y1),
            0,
            alto - 1
        )
    )

    derecha = int(
        np.clip(
            round(ancho * x2),
            0,
            ancho - 1
        )
    )

    inferior = int(
        np.clip(
            round(alto * y2),
            0,
            alto - 1
        )
    )

    cv2.rectangle(
        mascara,
        (
            izquierda,
            superior
        ),
        (
            derecha,
            inferior
        ),
        255,
        -1
    )


def crear_zona_textos(
    alto,
    ancho
):
    zona = np.zeros(
        (alto, ancho),
        dtype=np.uint8
    )

    cajas = [
        (0.015, 0.015, 0.270, 0.160),
        (0.015, 0.170, 0.105, 0.260),
        (0.015, 0.270, 0.080, 0.390),
        (0.730, 0.015, 0.990, 0.165),
        (0.010, 0.615, 0.085, 0.740),
        (0.010, 0.780, 0.260, 0.995),
        (0.855, 0.760, 0.995, 0.995)
    ]

    for caja in cajas:
        agregar_rectangulo_relativo(
            zona,
            *caja
        )

    return zona


def crear_zona_brujula(
    alto,
    ancho
):
    zona = np.zeros(
        (alto, ancho),
        dtype=np.uint8
    )

    agregar_rectangulo_relativo(
        zona,
        0.385,
        0.005,
        0.615,
        0.155
    )

    return zona


def crear_zona_crosshair(
    alto,
    ancho
):
    zona = np.zeros(
        (alto, ancho),
        dtype=np.uint8
    )

    agregar_rectangulo_relativo(
        zona,
        0.350,
        0.295,
        0.650,
        0.750
    )

    return zona


def crear_corredores_crosshair(
    alto,
    ancho
):
    zona = np.zeros(
        (alto, ancho),
        dtype=np.uint8
    )

    centro_x = ancho // 2
    centro_y = alto // 2

    # Corredor horizontal.
    cv2.rectangle(
        zona,
        (
            centro_x - int(ancho * 0.160),
            centro_y - int(alto * 0.085)
        ),
        (
            centro_x + int(ancho * 0.160),
            centro_y + int(alto * 0.085)
        ),
        255,
        -1
    )

    # Corredor vertical.
    cv2.rectangle(
        zona,
        (
            centro_x - int(ancho * 0.065),
            centro_y - int(alto * 0.210)
        ),
        (
            centro_x + int(ancho * 0.065),
            centro_y + int(alto * 0.210)
        ),
        255,
        -1
    )

    # Nucleo central.
    agregar_rectangulo_relativo(
        zona,
        0.385,
        0.400,
        0.615,
        0.610
    )

    # Cuatro regiones de esquinas.
    cajas = [
        (0.355, 0.300, 0.480, 0.485),
        (0.520, 0.300, 0.645, 0.485),
        (0.355, 0.515, 0.480, 0.725),
        (0.520, 0.515, 0.645, 0.725)
    ]

    for caja in cajas:
        agregar_rectangulo_relativo(
            zona,
            *caja
        )

    return zona


def crear_zonas_inferiores(
    alto,
    ancho
):
    zonas = []

    cajas = [
        (0.010, 0.790, 0.265, 0.995),
        (0.360, 0.790, 0.505, 0.995),
        (0.495, 0.790, 0.645, 0.995)
    ]

    for caja in cajas:
        zona = np.zeros(
            (alto, ancho),
            dtype=np.uint8
        )

        agregar_rectangulo_relativo(
            zona,
            *caja
        )

        zonas.append(
            zona
        )

    return zonas


def combinar_mascaras(*mascaras):
    resultado = np.zeros_like(
        mascaras[0]
    )

    for mascara in mascaras:
        resultado = cv2.bitwise_or(
            resultado,
            mascara
        )

    return resultado


def expandir_mascara(
    mascara,
    radio
):
    radio = max(
        0,
        int(radio)
    )

    if radio == 0:
        return mascara.copy()

    tamano = radio * 2 + 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            tamano,
            tamano
        )
    )

    return cv2.dilate(
        mascara,
        kernel,
        iterations=1
    )


def crecer_conectado(
    semillas,
    candidatos,
    radio
):
    resultado = semillas.copy()

    kernel = cv2.getStructuringElement(
        cv2.MORPH_CROSS,
        (3, 3)
    )

    for _ in range(int(radio)):
        expandida = cv2.dilate(
            resultado,
            kernel,
            iterations=1
        )

        nuevos = cv2.bitwise_and(
            expandida,
            candidatos
        )

        siguiente = cv2.bitwise_or(
            resultado,
            nuevos
        )

        if np.array_equal(
            siguiente,
            resultado
        ):
            break

        resultado = siguiente

    return resultado


def cerrar_direccional(
    mascara,
    tamano
):
    tamano = convertir_a_impar(
        tamano
    )

    horizontal = cv2.morphologyEx(
        mascara,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (
                tamano,
                1
            )
        ),
        iterations=1
    )

    vertical = cv2.morphologyEx(
        mascara,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (
                1,
                tamano
            )
        ),
        iterations=1
    )

    return combinar_mascaras(
        mascara,
        horizontal,
        vertical
    )


def banco_morfologico_lineas(
    candidatos,
    zona
):
    candidatos = cv2.bitwise_and(
        candidatos,
        zona
    )

    horizontales = np.zeros_like(
        candidatos
    )

    verticales = np.zeros_like(
        candidatos
    )

    for longitud in LONGITUDES_LINEA:
        longitud = convertir_a_impar(
            longitud
        )

        cierre_horizontal = cv2.morphologyEx(
            candidatos,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (
                    longitud,
                    1
                )
            ),
            iterations=1
        )

        cierre_vertical = cv2.morphologyEx(
            candidatos,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (
                    1,
                    longitud
                )
            ),
            iterations=1
        )

        apertura_horizontal = cv2.morphologyEx(
            cierre_horizontal,
            cv2.MORPH_OPEN,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (
                    max(
                        3,
                        longitud // 2
                    ),
                    1
                )
            ),
            iterations=1
        )

        apertura_vertical = cv2.morphologyEx(
            cierre_vertical,
            cv2.MORPH_OPEN,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (
                    1,
                    max(
                        3,
                        longitud // 2
                    )
                )
            ),
            iterations=1
        )

        horizontales = cv2.bitwise_or(
            horizontales,
            apertura_horizontal
        )

        verticales = cv2.bitwise_or(
            verticales,
            apertura_vertical
        )

    esquinas = cv2.bitwise_or(
        cv2.bitwise_and(
            horizontales,
            expandir_mascara(
                verticales,
                4
            )
        ),
        cv2.bitwise_and(
            verticales,
            expandir_mascara(
                horizontales,
                4
            )
        )
    )

    geometria = combinar_mascaras(
        horizontales,
        verticales,
        esquinas
    )

    return cv2.bitwise_and(
        geometria,
        candidatos
    )


def detectar_lineas_lsd(
    candidatos,
    zona
):
    resultado = np.zeros_like(
        candidatos
    )

    if not USAR_LSD:
        return resultado

    if not hasattr(
        cv2,
        "createLineSegmentDetector"
    ):
        return resultado

    entrada = cv2.bitwise_and(
        candidatos,
        zona
    )

    detector = cv2.createLineSegmentDetector(
        cv2.LSD_REFINE_STD
    )

    lineas = detector.detect(
        entrada
    )[0]

    if lineas is None:
        return resultado

    for linea in lineas[:, 0]:
        x1, y1, x2, y2 = map(
            float,
            linea
        )

        diferencia_x = x2 - x1
        diferencia_y = y2 - y1

        longitud = float(
            np.hypot(
                diferencia_x,
                diferencia_y
            )
        )

        if longitud < LONGITUD_MINIMA_LSD:
            continue

        if longitud > LONGITUD_MAXIMA_LSD:
            continue

        angulo = abs(float(
            np.degrees(
                np.arctan2(
                    diferencia_y,
                    diferencia_x
                )
            )
        ))

        es_horizontal = (
            angulo <= TOLERANCIA_ANGULO
            or angulo
            >= 180 - TOLERANCIA_ANGULO
        )

        es_vertical = (
            abs(angulo - 90)
            <= TOLERANCIA_ANGULO
        )

        if (
            not es_horizontal
            and not es_vertical
        ):
            continue

        cv2.line(
            resultado,
            (
                int(round(x1)),
                int(round(y1))
            ),
            (
                int(round(x2)),
                int(round(y2))
            ),
            255,
            3
        )

    # LSD solamente ayuda a medir líneas donde ya hay color verde.
    resultado = cv2.bitwise_and(
        expandir_mascara(
            resultado,
            1
        ),
        expandir_mascara(
            entrada,
            2
        )
    )

    return cv2.bitwise_and(
        resultado,
        zona
    )


def crear_evidencia_crosshair(
    frame,
    zona
):
    # Esta evidencia solamente se usa para calcular la apertura.
    # Ninguno de estos píxeles pasa directamente a la máscara final.

    fuerte = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_FUERTE
        ),
        zona
    )

    medio = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_MEDIO
        ),
        zona
    )

    tenue = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_TENUE
        ),
        zona
    )

    señal = combinar_mascaras(
        fuerte,
        medio,
        tenue
    )

    morfologia = banco_morfologico_lineas(
        señal,
        zona
    )

    lsd = detectar_lineas_lsd(
        señal,
        zona
    )

    geometria = combinar_mascaras(
        morfologia,
        lsd
    )

    evidencia_media = cv2.bitwise_and(
        medio,
        expandir_mascara(
            geometria,
            2
        )
    )

    evidencia_tenue = cv2.bitwise_and(
        tenue,
        geometria
    )

    return combinar_mascaras(
        fuerte,
        evidencia_media,
        evidencia_tenue
    )


def crear_mascara_estricta(
    frame,
    zona
):
    # Para brújula e indicadores inferiores.
    # No utiliza verde tenue, CLAHE, LSD ni reconstrucción libre.

    fuerte = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_FUERTE
        ),
        zona
    )

    medio = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_MEDIO
        ),
        zona
    )

    candidatos = cv2.bitwise_and(
        medio,
        expandir_mascara(
            fuerte,
            RADIO_ANCLA_ESTRICTA
        )
    )

    mascara = crecer_conectado(
        fuerte,
        candidatos,
        RADIO_CRECIMIENTO_ESTRICTO
    )

    mascara = cerrar_direccional(
        mascara,
        CIERRE_ESTRICTO
    )

    tamano = convertir_a_impar(
        DILATACION_ESTRICTA
    )

    mascara = cv2.dilate(
        mascara,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                tamano,
                tamano
            )
        ),
        iterations=1
    )

    return cv2.bitwise_and(
        mascara,
        zona
    )

def completar_marcas_verticales_brujula(
    frame,
    mascara_brujula,
    zona_brujula
):
    medio = detectar_verde(
        frame,
        VERDE_MEDIO
    )

    medio = cv2.bitwise_and(
        medio,
        zona_brujula
    )

    marcas_verticales = np.zeros_like(
        medio
    )

    for longitud in (
        3,
        5,
        7,
        9,
        11
    ):
        kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (
                1,
                longitud
            )
        )

        apertura = cv2.morphologyEx(
            medio,
            cv2.MORPH_OPEN,
            kernel,
            iterations=1
        )

        marcas_verticales = cv2.bitwise_or(
            marcas_verticales,
            apertura
        )

    # Solo acepta marcas próximas a la brújula ya encontrada.
    soporte = expandir_mascara(
        mascara_brujula,
        12
    )

    marcas_verticales = cv2.bitwise_and(
        marcas_verticales,
        soporte
    )

    resultado = combinar_mascaras(
        mascara_brujula,
        marcas_verticales
    )

    resultado = cv2.dilate(
        resultado,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (3, 3)
        ),
        iterations=1
    )

    return cv2.bitwise_and(
        resultado,
        zona_brujula
    )

def crear_soporte_temporal(
    lista_mascaras,
    indice
):
    referencia = lista_mascaras[
        indice
    ]

    contador = np.zeros(
        referencia.shape,
        dtype=np.uint16
    )

    inicio = max(
        0,
        indice - RADIO_TEMPORAL
    )

    final = min(
        len(lista_mascaras),
        indice + RADIO_TEMPORAL + 1
    )

    tamano = (
        RADIO_ALINEACION
        * 2
        + 1
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            tamano,
            tamano
        )
    )

    for vecino in range(
        inicio,
        final
    ):
        if vecino == indice:
            continue

        mascara_vecina = cv2.dilate(
            lista_mascaras[vecino],
            kernel,
            iterations=1
        )

        contador += (
            mascara_vecina > 0
        ).astype(np.uint16)

    return (
        contador >= MINIMO_VECINOS
    ).astype(np.uint8) * 255

def encontrar_vertice_esquina(
    evidencia,
    signo_x,
    signo_y,
    apertura_anterior
):
    alto, ancho = evidencia.shape

    centro_x = ancho // 2
    centro_y = alto // 2

    distancia_x_minima = int(
        ancho * DX_MINIMO
    )

    distancia_x_maxima = int(
        ancho * DX_MAXIMO
    )

    distancia_y_minima = int(
        alto * DY_MINIMO
    )

    distancia_y_maxima = int(
        alto * DY_MAXIMO
    )

    posicion_x_1 = (
        centro_x
        + signo_x * distancia_x_minima
    )

    posicion_x_2 = (
        centro_x
        + signo_x * distancia_x_maxima
    )

    posicion_y_1 = (
        centro_y
        + signo_y * distancia_y_minima
    )

    posicion_y_2 = (
        centro_y
        + signo_y * distancia_y_maxima
    )

    izquierda = max(
        0,
        min(posicion_x_1, posicion_x_2)
    )

    derecha = min(
        ancho,
        max(posicion_x_1, posicion_x_2) + 1
    )

    superior = max(
        0,
        min(posicion_y_1, posicion_y_2)
    )

    inferior = min(
        alto,
        max(posicion_y_1, posicion_y_2) + 1
    )

    region = evidencia[
        superior:inferior,
        izquierda:derecha
    ]

    if region.size == 0:
        return None

    # Une interrupciones muy pequeñas, pero no rellena la región.
    region = cerrar_direccional(
        region,
        3
    )

    posiciones_y, posiciones_x = np.where(
        region > 0
    )

    if len(posiciones_x) < MINIMO_PIXELES_ESQUINA:
        return None

    mejor_resultado = None
    mejor_puntuacion = -1e9

    longitud = LONGITUD_BUSQUEDA_ESQUINA
    grosor = GROSOR_BUSQUEDA_ESQUINA

    # Si existe apertura anterior, se utiliza solo como preferencia.
    if apertura_anterior is not None:
        esperado_x = (
            centro_x
            + signo_x * apertura_anterior[0]
        )

        esperado_y = (
            centro_y
            + signo_y * apertura_anterior[1]
        )
    else:
        esperado_x = None
        esperado_y = None

    # No hace falta probar cada píxel cuando hay demasiados.
    cantidad = len(posiciones_x)

    if cantidad > 800:
        paso = max(
            1,
            cantidad // 800
        )

        posiciones_x = posiciones_x[::paso]
        posiciones_y = posiciones_y[::paso]

    for posicion_local_x, posicion_local_y in zip(
        posiciones_x,
        posiciones_y
    ):
        posicion_x = (
            int(posicion_local_x)
            + izquierda
        )

        posicion_y = (
            int(posicion_local_y)
            + superior
        )

        # Las patas de las L se extienden hacia el interior.
        extremo_horizontal_x = (
            posicion_x
            - signo_x * longitud
        )

        extremo_vertical_y = (
            posicion_y
            - signo_y * longitud
        )

        horizontal_x_1 = max(
            0,
            min(
                posicion_x,
                extremo_horizontal_x
            )
        )

        horizontal_x_2 = min(
            ancho,
            max(
                posicion_x,
                extremo_horizontal_x
            ) + 1
        )

        horizontal_y_1 = max(
            0,
            posicion_y - grosor
        )

        horizontal_y_2 = min(
            alto,
            posicion_y + grosor + 1
        )

        vertical_x_1 = max(
            0,
            posicion_x - grosor
        )

        vertical_x_2 = min(
            ancho,
            posicion_x + grosor + 1
        )

        vertical_y_1 = max(
            0,
            min(
                posicion_y,
                extremo_vertical_y
            )
        )

        vertical_y_2 = min(
            alto,
            max(
                posicion_y,
                extremo_vertical_y
            ) + 1
        )

        apoyo_horizontal = np.count_nonzero(
            evidencia[
                horizontal_y_1:horizontal_y_2,
                horizontal_x_1:horizontal_x_2
            ]
        )

        apoyo_vertical = np.count_nonzero(
            evidencia[
                vertical_y_1:vertical_y_2,
                vertical_x_1:vertical_x_2
            ]
        )

        # Una L real debe tener apoyo en ambas direcciones.
        apoyo_comun = min(
            apoyo_horizontal,
            apoyo_vertical
        )

        puntuacion = (
            apoyo_comun * 5
            + apoyo_horizontal
            + apoyo_vertical
        )

        # La posición anterior ayuda, pero no obliga a conservar
        # siempre la misma apertura.
        if esperado_x is not None:
            distancia_anterior = np.hypot(
                posicion_x - esperado_x,
                posicion_y - esperado_y
            )

            puntuacion -= (
                distancia_anterior * 0.08
            )

        if puntuacion > mejor_puntuacion:
            mejor_puntuacion = puntuacion

            mejor_resultado = (
                posicion_x,
                posicion_y,
                apoyo_horizontal,
                apoyo_vertical
            )

    if mejor_resultado is None:
        return None

    posicion_x, posicion_y, apoyo_h, apoyo_v = (
        mejor_resultado
    )

    # Evita aceptar una raya aislada como una esquina.
    if (
        apoyo_h < MINIMO_PIXELES_ESQUINA
        or apoyo_v < MINIMO_PIXELES_ESQUINA
    ):
        return None

    return (
        abs(posicion_x - centro_x),
        abs(posicion_y - centro_y),
        posicion_x,
        posicion_y,
        apoyo_h + apoyo_v
    )

def estimar_apertura(
    evidencia,
    apertura_anterior
):
    alto, ancho = evidencia.shape

    distancia_x_minima = int(
        ancho * DX_MINIMO
    )

    distancia_x_maxima = int(
        ancho * DX_MAXIMO
    )

    distancia_y_minima = int(
        alto * DY_MINIMO
    )

    distancia_y_maxima = int(
        alto * DY_MAXIMO
    )

    propuestas = []

    for signo_x, signo_y in [
        (-1, -1),
        (1, -1),
        (-1, 1),
        (1, 1)
    ]:
        propuesta = encontrar_vertice_esquina(
            evidencia,
            signo_x,
            signo_y,
            apertura_anterior
        )

        if propuesta is not None:
            propuestas.append(
                propuesta
            )

    if propuestas:
        # Usa mediana para que una esquina falsa no desplace
        # por completo las otras tres.
        distancia_x = int(
            np.median(
                [
                    propuesta[0]
                    for propuesta in propuestas
                ]
            )
        )

        distancia_y = int(
            np.median(
                [
                    propuesta[1]
                    for propuesta in propuestas
                ]
            )
        )

        distancia_x = int(
            np.clip(
                distancia_x,
                distancia_x_minima,
                distancia_x_maxima
            )
        )

        distancia_y = int(
            np.clip(
                distancia_y,
                distancia_y_minima,
                distancia_y_maxima
            )
        )

        if apertura_anterior is not None:
            cambio_maximo_x = max(
                2,
                int(
                    ancho * MAXIMO_CAMBIO_X
                )
            )

            cambio_maximo_y = max(
                2,
                int(
                    alto * MAXIMO_CAMBIO_Y
                )
            )

            distancia_x = int(
                np.clip(
                    distancia_x,
                    apertura_anterior[0]
                    - cambio_maximo_x,
                    apertura_anterior[0]
                    + cambio_maximo_x
                )
            )

            distancia_y = int(
                np.clip(
                    distancia_y,
                    apertura_anterior[1]
                    - cambio_maximo_y,
                    apertura_anterior[1]
                    + cambio_maximo_y
                )
            )

            distancia_x = int(round(
                PESO_APERTURA_ANTERIOR
                * apertura_anterior[0]
                + (
                    1.0
                    - PESO_APERTURA_ANTERIOR
                )
                * distancia_x
            ))

            distancia_y = int(round(
                PESO_APERTURA_ANTERIOR
                * apertura_anterior[1]
                + (
                    1.0
                    - PESO_APERTURA_ANTERIOR
                )
                * distancia_y
            ))

        return (
            (
                distancia_x,
                distancia_y
            ),
            len(propuestas),
            propuestas
        )

    if apertura_anterior is not None:
        return (
            apertura_anterior,
            0,
            []
        )

    apertura_inicial = (
        int(
            ancho * DX_INICIAL
        ),
        int(
            alto * DY_INICIAL
        )
    )

    return (
        apertura_inicial,
        0,
        []
    )


def dibujar_crosshair(
    forma,
    apertura
):
    alto, ancho = forma

    centro_x = ancho // 2
    centro_y = alto // 2

    distancia_x, distancia_y = apertura

    longitud_horizontal_esquina = max(
        7,
        int(
            ancho
            * LONGITUD_PATA_HORIZONTAL_RELATIVA
        )
    )

    longitud_vertical_esquina = max(
        7,
        int(
            alto
            * LONGITUD_PATA_VERTICAL_RELATIVA
        )
    )

    mascara = np.zeros(
        forma,
        dtype=np.uint8
    )

    # --------------------------------------------------------------
    # Cuatro esquinas internas L
    # --------------------------------------------------------------

    for signo_x, signo_y in [
        (-1, -1),
        (1, -1),
        (-1, 1),
        (1, 1)
    ]:
        posicion_x = (
            centro_x
            + signo_x * distancia_x
        )

        posicion_y = (
            centro_y
            + signo_y * distancia_y
        )

        # Pata horizontal hacia el interior.
        cv2.line(
            mascara,
            (
                posicion_x,
                posicion_y
            ),
            (
                posicion_x
                - signo_x
                * longitud_horizontal_esquina,
                posicion_y
            ),
            255,
            GROSOR_PLANTILLA
        )

        # Pata vertical hacia el interior.
        cv2.line(
            mascara,
            (
                posicion_x,
                posicion_y
            ),
            (
                posicion_x,
                posicion_y
                - signo_y
                * longitud_vertical_esquina
            ),
            255,
            GROSOR_PLANTILLA
        )

    # --------------------------------------------------------------
    # X central
    # --------------------------------------------------------------

    radio_x = RADIO_X_CENTRAL

    cv2.line(
        mascara,
        (
            centro_x - radio_x,
            centro_y - radio_x
        ),
        (
            centro_x + radio_x,
            centro_y + radio_x
        ),
        255,
        GROSOR_X_CENTRAL
    )

    cv2.line(
        mascara,
        (
            centro_x - radio_x,
            centro_y + radio_x
        ),
        (
            centro_x + radio_x,
            centro_y - radio_x
        ),
        255,
        GROSOR_X_CENTRAL
    )

    # --------------------------------------------------------------
    # Cuatro brazos desde el centro
    # --------------------------------------------------------------

    longitud_horizontal = max(
        12,
        int(
            distancia_x
            * PROPORCION_BRAZO_HORIZONTAL
        )
    )

    longitud_vertical = max(
        12,
        int(
            distancia_y
            * PROPORCION_BRAZO_VERTICAL
        )
    )

    cv2.line(
        mascara,
        (
            centro_x - HUECO_CENTRAL,
            centro_y
        ),
        (
            centro_x - longitud_horizontal,
            centro_y
        ),
        255,
        GROSOR_PLANTILLA
    )

    cv2.line(
        mascara,
        (
            centro_x + HUECO_CENTRAL,
            centro_y
        ),
        (
            centro_x + longitud_horizontal,
            centro_y
        ),
        255,
        GROSOR_PLANTILLA
    )

    cv2.line(
        mascara,
        (
            centro_x,
            centro_y - HUECO_CENTRAL
        ),
        (
            centro_x,
            centro_y - longitud_vertical
        ),
        255,
        GROSOR_PLANTILLA
    )

    cv2.line(
        mascara,
        (
            centro_x,
            centro_y + HUECO_CENTRAL
        ),
        (
            centro_x,
            centro_y + longitud_vertical
        ),
        255,
        GROSOR_PLANTILLA
    )

    # --------------------------------------------------------------
    # Dos rayas verticales laterales dentro de las esquinas
    # --------------------------------------------------------------

    desplazamiento_lateral = int(
        distancia_x
        * POSICION_MARCA_LATERAL
    )

    semilongitud_marca = max(
        4,
        int(
            distancia_y
            * LONGITUD_MARCA_LATERAL_RELATIVA
        )
    )

    posicion_izquierda = (
        centro_x
        - desplazamiento_lateral
    )

    posicion_derecha = (
        centro_x
        + desplazamiento_lateral
    )

    cv2.line(
        mascara,
        (
            posicion_izquierda,
            centro_y - semilongitud_marca
        ),
        (
            posicion_izquierda,
            centro_y + semilongitud_marca
        ),
        255,
        GROSOR_MARCA_LATERAL
    )

    cv2.line(
        mascara,
        (
            posicion_derecha,
            centro_y - semilongitud_marca
        ),
        (
            posicion_derecha,
            centro_y + semilongitud_marca
        ),
        255,
        GROSOR_MARCA_LATERAL
    )

    return mascara


def crear_mascara_crosshair(
    evidencia,
    soporte_temporal,
    zona,
    apertura_anterior
):
    # La evidencia apoyada por frames vecinos es más confiable.
    evidencia_temporal = cv2.bitwise_and(
        evidencia,
        soporte_temporal
    )

    evidencia_temporal = cerrar_direccional(
        evidencia_temporal,
        3
    )

    # Si el apoyo temporal contiene suficiente información,
    # se utiliza para estimar la apertura.
    if np.count_nonzero(evidencia_temporal) >= 6:
        evidencia_medicion = evidencia_temporal
    else:
        evidencia_medicion = evidencia

    (
        apertura,
        esquinas_detectadas,
        propuestas
    ) = estimar_apertura(
        evidencia_medicion,
        apertura_anterior
    )

    # Importante:
    # La evidencia solo sirve para calcular la apertura.
    # No se copia a la máscara final.
    mascara = dibujar_crosshair(
        zona.shape,
        apertura
    )

    mascara = expandir_mascara(
        mascara,
        MARGEN_PLANTILLA
    )

    tamano = convertir_a_impar(
        DILATACION_CROSSHAIR
    )

    mascara = cv2.dilate(
        mascara,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                tamano,
                tamano
            )
        ),
        iterations=1
    )

    mascara = cv2.bitwise_and(
        mascara,
        zona
    )

    return (
        mascara,
        apertura,
        esquinas_detectadas,
        propuestas
    )


def crear_atlas_texto(
    ruta_video,
    duracion_video
):
    cantidad_muestras = min(
        MUESTRAS_ATLAS_TEXTO,
        max(
            1,
            int(
                duracion_video
                * FPS_SALIDA
            )
        )
    )

    tiempos = np.linspace(
        0.0,
        max(
            0.0,
            duracion_video - 0.05
        ),
        cantidad_muestras
    )

    captura = cv2.VideoCapture(
        str(ruta_video)
    )

    if not captura.isOpened():
        raise RuntimeError(
            "No se pudo abrir el video para crear el atlas."
        )

    acumulador = None
    muestras_validas = 0

    for indice, tiempo in enumerate(
        tiempos
    ):
        captura.set(
            cv2.CAP_PROP_POS_MSEC,
            float(tiempo) * 1000.0
        )

        correcto, frame = captura.read()

        if not correcto:
            continue

        fuerte = detectar_verde(
            frame,
            VERDE_FUERTE
        )

        alto, ancho = fuerte.shape

        mascara = cv2.bitwise_and(
            fuerte,
            crear_zona_textos(
                alto,
                ancho
            )
        )

        if acumulador is None:
            acumulador = np.zeros(
                mascara.shape,
                dtype=np.float32
            )

        acumulador += (
            mascara > 0
        ).astype(np.float32)

        muestras_validas += 1

        if (
            (indice + 1) % 50 == 0
            or indice + 1 == cantidad_muestras
        ):
            print(
                f"Atlas texto: "
                f"{indice + 1}/"
                f"{cantidad_muestras}"
            )

    captura.release()

    if (
        acumulador is None
        or muestras_validas == 0
    ):
        raise RuntimeError(
            "No se pudo crear el atlas de texto."
        )

    frecuencia = (
        acumulador
        / muestras_validas
    )

    return (
        frecuencia
        >= UMBRAL_ATLAS_TEXTO
    ).astype(np.uint8) * 255


def crear_mascara_texto(
    frame,
    atlas_texto
):
    fuerte = detectar_verde(
        frame,
        VERDE_FUERTE
    )

    medio = detectar_verde(
        frame,
        VERDE_MEDIO
    )

    alto, ancho = fuerte.shape

    zona = crear_zona_textos(
        alto,
        ancho
    )

    soporte = expandir_mascara(
        atlas_texto,
        RADIO_ATLAS_TEXTO
    )

    soporte = cv2.bitwise_and(
        soporte,
        zona
    )

    semillas = cv2.bitwise_and(
        fuerte,
        soporte
    )

    candidatos = cv2.bitwise_and(
        medio,
        soporte
    )

    mascara = crecer_conectado(
        semillas,
        candidatos,
        2
    )

    mascara = cerrar_direccional(
        mascara,
        CIERRE_TEXTO
    )

    tamano = convertir_a_impar(
        DILATACION_TEXTO
    )

    mascara = cv2.dilate(
        mascara,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                tamano,
                tamano
            )
        ),
        iterations=1
    )

    return cv2.bitwise_and(
        mascara,
        zona
    )


def extraer_frames(
    ruta_video,
    tiempos,
    fps_video,
    cantidad_frames
):
    captura = cv2.VideoCapture(
        str(ruta_video)
    )

    if not captura.isOpened():
        raise RuntimeError(
            "No se pudo abrir el video para extraer frames."
        )

    indice_inicial = int(
        np.clip(
            round(
                tiempos[0]
                * fps_video
            ),
            0,
            cantidad_frames - 1
        )
    )

    captura.set(
        cv2.CAP_PROP_POS_FRAMES,
        indice_inicial
    )

    indice_leido = indice_inicial - 1
    frame_actual = None
    frames = []

    for tiempo in tiempos:
        indice_objetivo = int(
            np.clip(
                round(
                    tiempo
                    * fps_video
                ),
                0,
                cantidad_frames - 1
            )
        )

        while indice_leido < indice_objetivo:
            correcto, frame_actual = captura.read()

            if not correcto:
                break

            indice_leido += 1

        if (
            frame_actual is None
            or indice_leido < indice_objetivo
        ):
            break

        frames.append(
            frame_actual.copy()
        )

    captura.release()

    return frames


def limpiar_archivos_png(carpeta):
    if not carpeta.exists():
        return

    for archivo in carpeta.glob(
        "*.png"
    ):
        archivo.unlink()


def guardar_png(
    ruta,
    imagen
):
    correcto = cv2.imwrite(
        str(ruta),
        imagen,
        [
            cv2.IMWRITE_PNG_COMPRESSION,
            COMPRESION_PNG
        ]
    )

    if not correcto:
        raise RuntimeError(
            f"No se pudo guardar: {ruta}"
        )


def crear_vista_previa(
    frame,
    mascara
):
    rojo = np.zeros_like(
        frame
    )

    rojo[:, :, 2] = 255

    mezcla = cv2.addWeighted(
        frame,
        0.60,
        rojo,
        0.40,
        0
    )

    salida = frame.copy()

    salida[
        mascara > 0
    ] = mezcla[
        mascara > 0
    ]

    return salida


def main():
    ruta_video = VIDEO

    if not ruta_video.is_file():
        raise SystemExit(
            f"No existe el video:\n{ruta_video}"
        )

    captura = cv2.VideoCapture(
        str(ruta_video)
    )

    if not captura.isOpened():
        raise SystemExit(
            f"No se pudo abrir el video:\n{ruta_video}"
        )

    (
        fps_video,
        cantidad_frames,
        duracion_video
    ) = obtener_datos_video(
        captura
    )

    captura.release()

    inicio, fin = calcular_intervalo(
        duracion_video
    )

    tiempos = crear_tiempos_salida(
        inicio,
        fin
    )

    cantidad_salida = len(
        tiempos
    )

    digitos = calcular_digitos(
        cantidad_salida
    )

    carpeta_video = (
        SALIDA
        / ruta_video.stem
    )

    carpeta_frames = (
        carpeta_video
        / "frames"
    )

    carpeta_mascaras = (
        carpeta_video
        / "masks"
    )

    carpeta_vistas = (
        carpeta_video
        / "vistas"
    )

    carpeta_diagnosticos = (
        carpeta_video
        / "diagnosticos"
    )

    for carpeta in (
        carpeta_frames,
        carpeta_mascaras,
        carpeta_vistas,
        carpeta_diagnosticos
    ):
        carpeta.mkdir(
            parents=True,
            exist_ok=True
        )

        if LIMPIAR_ANTERIORES:
            limpiar_archivos_png(
                carpeta
            )

    print()
    print("=" * 72)
    print(f"Video               : {ruta_video.name}")
    print(f"Duracion            : {duracion_video:.3f} s")
    print(f"Inicio              : {inicio:.3f} s")
    print(f"Final               : {fin:.3f} s")
    print(f"FPS salida          : {FPS_SALIDA:.3f}")
    print(f"Cantidad esperada   : {cantidad_salida}")
    print(f"Resultado           : {carpeta_video}")
    print("=" * 72)

    atlas_texto = crear_atlas_texto(
        ruta_video,
        duracion_video
    )

    if GUARDAR_ATLAS_TEXTO:
        guardar_png(
            carpeta_video
            / "atlas_texto.png",
            atlas_texto
        )

    frames = extraer_frames(
        ruta_video,
        tiempos,
        fps_video,
        cantidad_frames
    )

    if not frames:
        raise RuntimeError(
            "No se pudieron extraer frames."
        )

    preparados = []

    for indice, frame in enumerate(
        frames
    ):
        alto, ancho = frame.shape[:2]

        zona_crosshair = cv2.bitwise_and(
            crear_zona_crosshair(
                alto,
                ancho
            ),
            crear_corredores_crosshair(
                alto,
                ancho
            )
        )

        zona_brujula = crear_zona_brujula(
            alto,
            ancho
        )

        zonas_inferiores = crear_zonas_inferiores(
            alto,
            ancho
        )

        evidencia_crosshair = (
            crear_evidencia_crosshair(
                frame,
                zona_crosshair
            )
        )

        mascara_brujula = crear_mascara_estricta(
            frame,
            zona_brujula
        )
        
        mascara_brujula = completar_marcas_verticales_brujula(
            frame,
            mascara_brujula,
            zona_brujula
        )

        mascaras_inferiores = []

        for zona in zonas_inferiores:
            mascaras_inferiores.append(
                crear_mascara_estricta(
                    frame,
                    zona
                )
            )

        preparados.append({
            "frame": frame,
            "texto": crear_mascara_texto(
                frame,
                atlas_texto
            ),
            "zona_crosshair": zona_crosshair,
            "evidencia_crosshair": evidencia_crosshair,
            "brujula": mascara_brujula,
            "inferiores": mascaras_inferiores
        })

        if (
            (indice + 1) % 50 == 0
            or indice + 1 == len(frames)
        ):
            print(
                f"Preparados: "
                f"{indice + 1}/{len(frames)}"
            )

    evidencias_crosshair = [
        preparado["evidencia_crosshair"]
        for preparado in preparados
    ]

    apertura_anterior = None

    for indice, preparado in enumerate(
        preparados
    ):
        soporte_temporal = crear_soporte_temporal(
            evidencias_crosshair,
            indice
        )

        (
            mascara_crosshair,
            apertura_anterior,
            esquinas_detectadas,
            propuestas_esquinas
        ) = crear_mascara_crosshair(
            preparado["evidencia_crosshair"],
            soporte_temporal,
            preparado["zona_crosshair"],
            apertura_anterior
        )

        mascara_final = combinar_mascaras(
            preparado["texto"],
            preparado["brujula"],
            mascara_crosshair,
            *preparado["inferiores"]
        )

        _, mascara_final = cv2.threshold(
            mascara_final,
            127,
            255,
            cv2.THRESH_BINARY
        )

        nombre = (
            f"{indice:0{digitos}d}.png"
        )

        guardar_png(
            carpeta_frames
            / nombre,
            preparado["frame"]
        )

        guardar_png(
            carpeta_mascaras
            / nombre,
            mascara_final
        )

        if GUARDAR_VISTAS:
            vista = crear_vista_previa(
                preparado["frame"],
                mascara_final
            )

            guardar_png(
                carpeta_vistas
                / nombre,
                vista
            )

        if GUARDAR_DIAGNOSTICOS:
            diagnostico = preparado[
                "frame"
            ].copy()

            plantilla = dibujar_crosshair(
                preparado[
                    "zona_crosshair"
                ].shape,
                apertura_anterior
            )

            # Plantilla inferida en magenta.
            diagnostico[
                plantilla > 0
            ] = (
                255,
                0,
                255
            )

            # Vértices de esquinas realmente detectados en amarillo.
            for propuesta in propuestas_esquinas:
                posicion_x = int(
                    propuesta[2]
                )

                posicion_y = int(
                    propuesta[3]
                )

                cv2.circle(
                    diagnostico,
                    (
                        posicion_x,
                        posicion_y
                    ),
                    7,
                    (
                        0,
                        255,
                        255
                    ),
                    2
                )

            texto_diagnostico = (
                f"dx={apertura_anterior[0]} "
                f"dy={apertura_anterior[1]} "
                f"esquinas={esquinas_detectadas}"
            )

            cv2.putText(
                diagnostico,
                texto_diagnostico,
                (
                    20,
                    30
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.70,
                (
                    255,
                    255,
                    0
                ),
                2,
                cv2.LINE_AA
            )

            guardar_png(
                carpeta_diagnosticos
                / nombre,
                diagnostico
            )

        if (
            (indice + 1) % 50 == 0
            or indice + 1 == len(preparados)
        ):
            print(
                f"Guardadas: "
                f"{indice + 1}/{len(preparados)}"
            )

    print()
    print("=" * 72)
    print("Proceso finalizado correctamente.")
    print(f"Frames guardados     : {len(frames)}")
    print(f"Mascaras guardadas   : {len(frames)}")
    print(f"Resultado            : {carpeta_video}")
    print("=" * 72)


if __name__ == "__main__":
    main()